In [ ]:
# 1- Improt Libraries
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# 2 load data
df_raw = pd.read_csv('C:/Users/hnawaz1/Downloads/Pilot_MATLAB_Closed.csv')
print(df_raw.shape)
df_raw.head()

In [ ]:
#Inspect and Sanity Check
df = df_raw[['Item', 'Description', 'Request_Type_Normalized']].copy()

#Basic Checks
print ("Missing Description", df['Description'].isna().sum())
print ("Missing Label:", df['Request_Type_Normalized'].isna().sum())
print ("Missing Item:", df['Item'].isna().sum())
#Drop missing
df = df.dropna(subset=['Item', 'Description', 'Request_Type_Normalized'])

#Strip Whitespaces
df['Item'] = df['Item'].astype(str).str.strip()
df['Description'] = df['Description'].astype(str).str.strip()

# Remove very short junk description
df = df[df['Description'].str.len() >= 20]

#label distribution
df['Request_Type_Normalized'].value_counts()

In [ ]:
#4 Encode Labels (2 class majority)
#sorted() so that id 0 is always the same category every time we run the code

MAJORITY_LABEL = df["Request_Type_Normalized"].value_counts().idxmax()
print("Majority label detected:", MAJORITY_LABEL)

    #create 2 class column
df["Request_Type_Model"] = np.where(
    df["Request_Type_Normalized"] == MAJORITY_LABEL,
    MAJORITY_LABEL,
    "Other"
)

#create stabe label ids from 2-class column
label_names = sorted(df['Request_Type_Model'].unique())
label_to_id = {name:i for i, name in enumerate(label_names)}
id_to_label = {i:name for name,i in label_to_id.items()}

#create new column 
df['label_id'] = df['Request_Type_Model'].map(label_to_id)

print("Num classes:", len(label_names))
print("Mapping Preview:", label_to_id)
print(df["Request_Type_Model"].value_counts()),
df[["Request_Type_Normalized", "Request_Type_Model", "label_id"]].head()

In [ ]:
from sklearn.model_selection import train_test_split
# 5 Train/Val/Test Split
df["TextForModel"] = df["Item"].astype(str) + "|" + df["Description"].astype(str)
X = df["TextForModel"].values
y = df["label_id"].values

#80/20 split first
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#Split train into train/val (e.g 80.20 of train)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print("Train:",len(X_train), "Val:", len(X_val), "Test:", len (X_test))
## sanity check class distribution
print("Train dist:", np.bincount(y_train))
print("Val dist:", np.bincount(y_val))
print("Test dist:", np.bincount(y_test))
print("id_to_label:", id_to_label)

In [ ]:
Text Vectorization layer
max_tokens = 10000
output_sequence_length = 200

vectorize_layer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=output_sequence_length
)
# adapt only to training data
vectorize_layer.adapt(X_train)

In [ ]:
# Model building
num_classes = len(label_names)

model = tf.keras.Sequential([
    vectorize_layer,
    layers.Embedding(input_dim=max_tokens, output_dim=128),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')

In [ ]:
#Model compilation
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']

In [ ]:
# force text to clean strings (no NaN/None
import numpy as np

def to_1d_string_array(x):
    x = np.asarray(x).reshape(-1)
    
    x = np.array([("" if s is None else str(s)) for s in x], dtype=object)
    
    x = np.where(x == "None", "", x)
    x = np.where(x == "nan", "", x)
    x = np.where(x == "NaN", "", x)
   
    return x
    
X_train = to_1d_string_array(X_train)
X_val   = to_1d_string_array(X_val)
X_test  = to_1d_string_array(X_test)



y_train = np.asarray(y_train, dtype=np.int32). reshape(-1)
y_val   = np.asarray(y_val, dtype=np.int32). reshape(-1)
y_test  = np.asarray(y_test, dtype=np.int32). reshape(-1)

from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))
print("class_weight:", class_weight)




print("X_train dtype/shape:", X_train.dtype, X_train.shape, "sample:",X_train[0][:80])
print("y_train dtype/shape/classes:", y_train.dtype, y_train.shape, "classes:", np.unique(y_train))


In [ ]:
history = model.fit(
    X_train,  
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    class_weight=class_weight

In [ ]:
import matplotlib.pyplot as plt
h = history .history

plt.figure()
plt.plot(h["loss"], label="train_loss")
plt.plot(h["val_loss"], label="val_loss")
plt.title("loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure()
plt.plot(h["accuracy"], label="train_acc")
plt.plot(h["val_accuracy"], label="val_acc")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


In [ ]:
#then i tested with live enteries: import numpy as np
import tensorflow as tf

text = ", Floating License License File Change (CLS) software enhancements, sdf coffe training reports and more for many company applicationsclas cls clsl vls vehcile."

x= tf.constant([text], dtype=tf.string)
probs = model.predict(x, verbose=0)[0]
pred_id = int(np.argmax(probs))
conf = float(np.max(probs))

print("Predicted label:", id_to_label[pred_id], "| Confidence:", float(np.max(probs)))

pred_label = id_to_label[pred_id]
conf = float(np.max(probs))

final_label = pred_label if conf >=0.80 else "Needs Review"
print("Final:", final_label," | Pred:",pred_label,"| Confidence:", round(conf,3))

In [ ]:
# test
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print ("TEST accuracy:", test_acc, "TEST loss:", test_loss)

In [ ]:
#then i did the confusion matriximport numpy as np
from sklearn.metrics import classification_report, confusion_matrix
y_pred = np.argmax(model.predict(X_test), axis=1)

print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassfication report:\n", classification_report(y_test, y_pred, digits=3))